# Lesson 5: Evoked and induced activity

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

April 1, 2026


## Import the data

In [ ]:
import mne
import numpy as np
from matplotlib import pyplot as plt
folder_name = '/Users/dkleeva/Library/CloudStorage/GoogleDrive-dkleeva@gmail.com/My Drive/Teaching/Сигналы целого мозга 2026/Scripts/Data/EEG/'
raw = mne.io.read_raw_edf(folder_name + 'sounds.edf',preload=True)
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)
raw.filter(0.1, 40)

In [ ]:
%matplotlib qt
raw.plot()

## Events and epochs

In [ ]:
# events = mne.find_events(ra, stim_channel="STI 014")
events, event_id = mne.events_from_annotations(raw)
print(events)
print(event_id)

The first column contains the event onset (in samples) **with first_samp included**. The last column contains the event code. The second column contains the signal value of the immediately preceding sample, and reflects the fact that event arrays sometimes originate from analog voltage channels (“trigger channels” or “stim channels”). In most cases, the second column is all zeros and can be ignored.

In [ ]:
%matplotlib inline
fig = mne.viz.plot_events(
    events, sfreq=raw.info["sfreq"], first_samp=raw.first_samp, event_id=event_id)

In [ ]:
events = events[events[:, 2] == event_id[np.str_('Key Space')]]
print(events)


## Epochs

In [ ]:
epochs_long = mne.Epochs(raw, events, tmin=-0.5, tmax=0.7, baseline=(-0.1, 0), event_id={'Sound':1}, preload=True)
epochs=epochs_long.copy().crop(-0.1,0.7)

In [ ]:
epochs

In [ ]:
epochs.info

In [ ]:
print(epochs.drop_log)

In [ ]:
%matplotlib qt
epochs.plot()

In [ ]:
#remove bad epochs based on the rejection parameters
reject = dict(eeg=100e-6)
epochs.drop_bad(reject=reject)

print(epochs.drop_log)

In [ ]:
#Let's return to the original epochs
epochs = mne.Epochs(raw, events, tmin=-0.1, tmax=0.7, baseline=(-0.1, 0), event_id={'Sound':1}, preload=True)

## Clean the data

In [ ]:
%matplotlib qt
ica = mne.preprocessing.ICA(n_components=0.95, max_iter='auto', random_state=97)
ica.fit(epochs)
ica.plot_sources(epochs)


In [ ]:
%matplotlib inline
ica.plot_properties(epochs, picks=[0, 1, 2, 3])

In [ ]:
ica.exclude=[0]
ica.apply(epochs)
ica.apply(epochs_long)

In [ ]:
epochs.plot()
plt.show()

In [ ]:
#remove bad epochs based on the rejection parameters
reject = dict(eeg=100e-6)
epochs.drop_bad(reject=reject)

print(epochs.drop_log)

## ERP

In [ ]:
%matplotlib inline
epochs.plot_image(combine="mean", picks=['Cz', 'C3', 'C4', 'FCz', 'Fz'])
plt.show()

In [ ]:
evoked = epochs.average()

In [ ]:
%matplotlib qt
evoked.plot()

In [ ]:
epochs.average().plot_joint([0.06])

In [ ]:
epochs[10].average().plot_joint([0.06])

In [ ]:
%matplotlib inline
evoked.plot(spatial_colors=True, gfp=True)
plt.show()

In [ ]:
evoked.plot_topomap(times=np.linspace(-0.1, 0.4, 5), colorbar=True)
plt.show()

In [ ]:
%matplotlib qt
mne.viz.plot_evoked_topo(evoked)

In [ ]:
subjects_dir = '/Users/dkleeva/mne_data/MNE-fsaverage-data/'
maps = mne.make_field_map(
    evoked,
    trans='fsaverage',
    subject="fsaverage",
    subjects_dir=subjects_dir,
    origin="auto",
)
evoked.plot_field(maps, time=0.1)

## Change the reference 

In [ ]:
epochs.set_eeg_reference('average')
epochs_long.set_eeg_reference('average')

In [ ]:
%matplotlib inline
evoked=epochs.average()
evoked.plot_joint()
plt.show()


In [ ]:
epochs.plot_image(combine="mean", picks=['Cz', 'C3', 'C4', 'FCz', 'Fz'])
plt.show()

In [ ]:
evoked_csd = mne.preprocessing.compute_current_source_density(evoked)
evoked.plot_joint([0.05], title="Average Reference", show=False)
evoked_csd.plot_joint([0.05], title="Current Source Density")
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 4, layout="constrained")
fig.set_size_inches(10, 10)
for i, lambda2 in enumerate([0, 1e-7, 1e-5, 1e-3]):
    for j, m in enumerate([5, 4, 3, 2]):
        this_evoked_csd = mne.preprocessing.compute_current_source_density(
            evoked, stiffness=m, lambda2=lambda2
        )
        this_evoked_csd.plot_topomap(
            0.05, axes=ax[i, j], contours=4, time_unit="s", colorbar=False, show=False
        )
        ax[i, j].set_title(f"stiffness={m}\nλ²={lambda2}")

## Projector to the space of the signal

In [ ]:
ev_sig = epochs.average().copy().crop(0.05,0.06)
projs = mne.compute_proj_evoked(ev_sig, n_eeg=1)  # dominant spatial pattern

In [ ]:
%matplotlib inline
mne.viz.plot_projs_topomap(projs, info=raw.info)
plt.show()


In [ ]:
proj_data = projs[0]['data']['data']

In [ ]:
proj_data.shape

In [ ]:
P = proj_data.T @ proj_data 
plt.imshow(P)

In [ ]:
X_ep = epochs.get_data()
X_proj = np.array([P @ ep for ep in X_ep])


In [ ]:
epochs_proj = mne.EpochsArray(X_proj, epochs.info, tmin=epochs.tmin)

In [ ]:
ev_before = epochs.average()
ev_after = epochs_proj.average()


In [ ]:
%matplotlib inline
ev_before.plot_joint([0.05])
ev_after.plot_joint([0.05])
plt.show()

In [ ]:
%matplotlib qt
epochs_proj.plot()

In [ ]:
%matplotlib inline
epochs[17].average().plot_joint([0.03])
plt.show()
epochs_proj[17].average().plot_joint([0.03])
plt.show()


## Dangers of filtering

In [ ]:
%matplotlib inline
epochs.average().filter(10,20).plot_joint([0.05])
plt.show()

In [ ]:
%matplotlib inline
epochs.average().filter(5,30).plot_joint([0.05])
plt.show()

In [ ]:
# take channel cz and show original signal of average evoked and filtered signal of average evoked overlayed with different cutoffs
#use matplotlib 

original_signal = np.squeeze(evoked.copy().pick('Cz').data)
filtered_signal = np.squeeze(evoked.copy().filter(0,2).pick('Cz').data)

plt.plot(epochs.times, original_signal, label='Original')
plt.plot(epochs.times, filtered_signal, label='Filtered')
plt.legend()
plt.show()


## Time-frequency analysis

In [ ]:
epochs_long.average().plot_joint([0.05])
plt.show()

In [ ]:
freqs = np.logspace(*np.log10([3, 30]), num=50)
n_cycles = freqs / 2.0  
power = epochs_long.compute_tfr(
    method="morlet",
    freqs=freqs,
    n_cycles=n_cycles,
    average=False,
    return_itc=False
)

In [ ]:
# power.apply_baseline((-0.5, 0), mode='zscore')

In [ ]:
def log_baseline(power, baseline=(-0.5, 0.0)):
    """Log-domain baseline correction for an MNE TFR object.

    Parameters
    ----------
    power : mne.time_frequency.BaseTFR
        TFR object (e.g., EpochsTFR/AverageTFR).
    baseline : tuple[float, float]
        Baseline interval in seconds.

    Returns
    -------
    power_bc : same type as `power`
        Baseline-corrected TFR object (copy of input).
    """
    power_bc = power.copy()
    times = power_bc.times
    data = power_bc.data

    mask = (times >= baseline[0]) & (times <= baseline[1])
    if not mask.any():
        raise ValueError(
            f"Baseline {baseline} empty in [{times[0]:.3f}, {times[-1]:.3f}]"
        )

    log_pow = np.log(np.maximum(data, 1e-30))
    base_mean = log_pow[..., mask].mean(axis=-1, keepdims=True)
    power_bc.data = (log_pow - base_mean).astype(np.float32)

    return power_bc

In [ ]:
power = log_baseline(power, baseline=(-0.5, 0.0))

In [ ]:
power.average().plot_joint(timefreqs=[(0.01,5), (0.2, 7)])
plt.show()

In [ ]:
power.average().plot_topomap(fmin=5, fmax=7, tmin=0, tmax=0.01)
plt.show()
power.average().plot_topomap(fmin=13, fmax=25, tmin=0.2, tmax=0.21)
plt.show()


In [ ]:
%matplotlib qt
power.average().plot_topo()

In [ ]:
#now subtract evoked

**Intertrial coherence (ITC)** is a measure of how consistent oscillatory phase is across an ensemble of trials. How much do different trials resemble each other?

In [ ]:
freqs = np.logspace(*np.log10([3, 30]), num=50)
n_cycles = freqs / 2.0  
_, itc = epochs_long.compute_tfr(
    method="morlet",
    freqs=freqs,
    n_cycles=n_cycles,
    average=True,
    return_itc=True
)

In [ ]:
itc.plot_topo()